# Projet Kayak — Partie 5 : Stockage des données brutes sur AWS S3

On uploade les **données brutes** du projet sur un bucket S3 :
- `weather.csv` : prévisions météo des 35 villes (sortie de l'API OpenWeatherMap)
- `hotels.jsonl` : 100 hôtels scrapés sur Booking via Scrapy + Playwright

**Pourquoi S3 ?** S3 est un service de stockage objet d'AWS, parfait pour conserver des fichiers bruts qui pourront être rechargés/retraités plus tard. C'est le pattern standard d'un **data lake** : on garde toujours la donnée brute, et les transformations propres vont dans une base SQL (partie 6).

**Sécurité** : on lit les credentials depuis `.env` (jamais en clair dans le code).

## 1. Imports et configuration

In [1]:
import os
import boto3
from botocore.exceptions import ClientError
from dotenv import load_dotenv

load_dotenv()

# On lit les variables d'environnement (depuis le .env)
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_DEFAULT_REGION", "eu-north-1")
S3_BUCKET = os.getenv("S3_BUCKET_NAME")

# Vérifications de sanité
assert AWS_ACCESS_KEY_ID, "AWS_ACCESS_KEY_ID manquant dans .env"
assert AWS_SECRET_ACCESS_KEY, "AWS_SECRET_ACCESS_KEY manquant dans .env"
assert S3_BUCKET, "S3_BUCKET_NAME manquant dans .env"

print(f"Région : {AWS_REGION}")
print(f"Bucket : {S3_BUCKET}")
print(f"Access Key (masquée) : {AWS_ACCESS_KEY_ID[:8]}...{AWS_ACCESS_KEY_ID[-4:]}")

Région : eu-west-1
Bucket : kayak-jedha-alex
Access Key (masquée) : AKIAUQTK...FJOA


## 2. Création du client S3

`boto3.client("s3")` crée un objet qui parle au service S3 d'AWS. C'est cet objet qu'on utilise pour toutes les opérations (upload, download, list, etc.).

In [2]:
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_REGION,
)

# Test de connexion : on liste les buckets accessibles avec ces credentials
try:
    response = s3.list_buckets()
    print("✅ Connexion AWS établie. Buckets accessibles :")
    for bucket in response["Buckets"]:
        marker = " ← celui qu'on utilise" if bucket["Name"] == S3_BUCKET else ""
        print(f"  - {bucket['Name']}{marker}")
except ClientError as e:
    print(f"❌ Erreur de connexion : {e}")

✅ Connexion AWS établie. Buckets accessibles :
  - airflow-citymapper-bucket
  - csvbucket-dbt-demo
  - kayak-jedha-alex ← celui qu'on utilise
  - mlflow-artifacts-votreentreprise
  - my-first-bk-bucket-123123
  - newbucket-cicd-lexi
  - s3-demo-bucket-k8s


## 3. Upload des fichiers

On définit une petite fonction utilitaire pour uploader un fichier local vers S3, avec un message clair à chaque étape.

In [3]:
def upload_to_s3(local_path: str, s3_key: str) -> None:
    """Uploade un fichier local vers S3.
    
    Args:
        local_path : chemin du fichier sur ta machine (ex: 'data/weather.csv')
        s3_key     : chemin dans le bucket (ex: 'raw/weather.csv')
    """
    file_size = os.path.getsize(local_path) / 1024  # en KB
    print(f"📤 Upload de {local_path} ({file_size:.1f} KB) → s3://{S3_BUCKET}/{s3_key}")
    s3.upload_file(local_path, S3_BUCKET, s3_key)
    print(f"   ✅ Terminé")

In [4]:
# On organise les uploads dans un dossier 'raw/' pour garder le bucket propre.
# C'est une convention courante : raw/ pour les données brutes, processed/ pour les nettoyées, etc.

upload_to_s3("data/weather.csv", "raw/weather.csv")
upload_to_s3("data/hotels.jsonl", "raw/hotels.jsonl")

📤 Upload de data/weather.csv (1.8 KB) → s3://kayak-jedha-alex/raw/weather.csv
   ✅ Terminé
📤 Upload de data/hotels.jsonl (56.5 KB) → s3://kayak-jedha-alex/raw/hotels.jsonl
   ✅ Terminé


## 4. Vérification : lister le contenu du bucket

On confirme que les fichiers sont bien arrivés en interrogeant S3.

In [5]:
response = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix="raw/")

if "Contents" in response:
    print(f"Contenu de s3://{S3_BUCKET}/raw/ :")
    for obj in response["Contents"]:
        size_kb = obj["Size"] / 1024
        modified = obj["LastModified"].strftime("%Y-%m-%d %H:%M:%S")
        print(f"  - {obj['Key']:<25} {size_kb:>8.1f} KB   {modified}")
else:
    print("⚠️  Bucket vide (le upload a-t-il fonctionné ?)")

Contenu de s3://kayak-jedha-alex/raw/ :
  - raw/hotels.jsonl              56.5 KB   2026-05-02 15:00:16
  - raw/weather.csv                1.8 KB   2026-05-02 15:00:15


## 5. (Optionnel) Test de re-téléchargement

Pour démontrer que les données sont bien accessibles, on télécharge un fichier depuis S3 et on l'ouvre. Utile pour le jury : montre que la donnée est *vraiment* là, pas juste qu'on a appelé l'API.

In [6]:
import pandas as pd
import io

# On télécharge le fichier directement en mémoire (sans le réécrire sur disque)
obj = s3.get_object(Bucket=S3_BUCKET, Key="raw/weather.csv")
df_from_s3 = pd.read_csv(io.BytesIO(obj["Body"].read()))

print(f"✅ Fichier téléchargé depuis S3 : {len(df_from_s3)} lignes, {len(df_from_s3.columns)} colonnes")
df_from_s3.head()

✅ Fichier téléchargé depuis S3 : 35 lignes, 7 colonnes


,city,lat,lon,temp_avg,rain_total,clouds_avg,weather_score
0,Mont Saint Michel,48.635954,-1.511460,15.2,10.4,77.5,2.25
1,Saint Malo,48.649518,-2.026041,14.6,8.8,75.1,2.69
2,Bayeux,49.276462,-0.702474,14.8,27.7,66.7,-5.72
3,Le Havre,49.493898,0.107973,14.7,26.5,47.3,-3.28
4,Rouen,49.440459,1.093966,15.1,13.8,50.2,3.18
